# UniProt 2026 → DeepPeptide-like dataset

Ноутбук рассчитан на запуск из папки `notebooks/`. Он автоматически поднимается на уровень выше и работает с:

```text
data/uniprot_2026/
  raw/uniprot_pep.json
  raw/uniprot_propep.json
  processed_data/
```

Выходы:
- `data/uniprot_2026/labeled_sequences.csv` — legacy DeepPeptide format
- `data/uniprot_2026/protein_sequences.fasta` и `protein_sequence.fasta`
- `data/uniprot_2026/protein_id_homo.txt`
- `data/uniprot_2026/processed_data/proteins_master.parquet`
- `data/uniprot_2026/processed_data/features_master.parquet`
- `data/uniprot_2026/processed_data/build_report.json`


In [1]:

import json
import gzip
import hashlib
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd


# Notebook лежит в notebooks/, поэтому идём на уровень выше.
# Если запускаешь не из notebooks/, код всё равно старается найти data/uniprot_2026.
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

DATA_DIR = PROJECT_ROOT / "data" / "uniprot_2026"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed_data"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PEP_JSON = RAW_DIR / "uniprot_pep.json"
PROPEP_JSON = RAW_DIR / "uniprot_propep.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("PEP_JSON exists:", PEP_JSON.exists(), PEP_JSON)
print("PROPEP_JSON exists:", PROPEP_JSON.exists(), PROPEP_JSON)


PROJECT_ROOT: /home/oskar/work/DeepPeptide
DATA_DIR: /home/oskar/work/DeepPeptide/data/uniprot_2026
PEP_JSON exists: True /home/oskar/work/DeepPeptide/data/uniprot_2026/raw/uniprot_pep.json
PROPEP_JSON exists: True /home/oskar/work/DeepPeptide/data/uniprot_2026/raw/uniprot_propep.json


In [2]:

MIN_FEATURE_LEN = 5
MAX_FEATURE_LEN = 50

TARGET_TYPES = {
    "Peptide": "PEPTIDE",
    "Propeptide": "PROPEP",
}

# В UniProt JSON обычно "Signal" и "Transit peptide"; оставляем запасные варианты.
TARGETING_FEATURE_TYPES = {
    "Signal",
    "Signal peptide",
    "Transit peptide",
    "Transit",
}

# DeepPeptide исключал sorting signals, размеченные как propeptides через эти PROSITE ProRules.
SORTING_SIGNAL_PRORULES = {"PRU00477", "PRU01070"}


def load_uniprot_json(path: Path):
    path = Path(path)
    if path.suffix == ".gz":
        with gzip.open(path, "rt", encoding="utf-8") as f:
            data = json.load(f)
    else:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

    if isinstance(data, dict) and "results" in data:
        return data["results"]

    if isinstance(data, list):
        return data

    raise ValueError(f"Unexpected UniProt JSON format: {path}")


def get_nested(obj, *keys, default=None):
    cur = obj
    for key in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key)
        if cur is None:
            return default
    return cur


def sequence_md5(seq: str) -> str:
    return hashlib.md5(seq.encode("utf-8")).hexdigest()


def get_location(feature):
    loc = feature.get("location", {}) or {}
    start_obj = loc.get("start", {}) or {}
    end_obj = loc.get("end", {}) or {}

    start = start_obj.get("value")
    end = end_obj.get("value")

    start_modifier = start_obj.get("modifier")
    end_modifier = end_obj.get("modifier")

    if start is not None:
        start = int(start)

    if end is not None:
        end = int(end)

    return start, end, start_modifier, end_modifier


def extract_evidence(feature):
    evidence_codes = []
    evidence_sources = []
    evidence_ids = []
    pubmed_ids = []

    for ev in feature.get("evidences", []) or []:
        code = ev.get("evidenceCode")
        source = ev.get("source")
        ev_id = ev.get("id")

        if code:
            evidence_codes.append(code)
        if source:
            evidence_sources.append(source)
        if ev_id:
            evidence_ids.append(ev_id)
        if source == "PubMed" and ev_id:
            pubmed_ids.append(ev_id)

    return evidence_codes, evidence_sources, evidence_ids, pubmed_ids


def flatten_strings(obj):
    """Рекурсивно достаём строки из dict/list, чтобы искать PRU00477/PRU01070."""
    out = []
    if isinstance(obj, dict):
        for v in obj.values():
            out.extend(flatten_strings(v))
    elif isinstance(obj, list):
        for v in obj:
            out.extend(flatten_strings(v))
    elif isinstance(obj, str):
        out.append(obj)
    return out


def feature_has_sorting_prorule(feature):
    haystack = " ".join(flatten_strings(feature))
    return any(rule in haystack for rule in SORTING_SIGNAL_PRORULES)


def intervals_cover_full_length(intervals, protein_length):
    """Проверяет, покрывает ли объединение интервалов 1..protein_length без дыр."""
    if not intervals or not protein_length:
        return False

    intervals = sorted((int(s), int(e)) for s, e in intervals if pd.notna(s) and pd.notna(e))
    if not intervals:
        return False

    current_end = 0
    for start, end in intervals:
        if start > current_end + 1:
            return False
        current_end = max(current_end, end)

    return current_end >= int(protein_length)


def parse_coords(coord_string):
    if pd.isna(coord_string):
        return []

    coord_string = str(coord_string).strip()
    if coord_string == "" or coord_string.lower() == "nan":
        return []

    out = []
    for part in coord_string.split(","):
        part = part.strip().lstrip("(").rstrip(")")
        if not part:
            continue
        start, end = part.split("-")
        out.append((int(start), int(end)))
    return out


def coords_to_string(features_df: pd.DataFrame, label: str) -> str:
    # DeepPeptide legacy style: "(97-101),(104-108)"
    sub = features_df[features_df["label"] == label].sort_values(["start", "end"])
    if sub.empty:
        return ""
    return ",".join(f"({int(r.start)}-{int(r.end)})" for r in sub.itertuples(index=False))


def binary_string(sequence: str, coords):
    arr = np.zeros(len(sequence), dtype=np.int8)
    for start, end in coords:
        arr[start - 1:end] = 1
    return "".join(arr.astype(str))


def write_fasta(df: pd.DataFrame, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        for row in df.itertuples(index=False):
            f.write(f">{row.protein_id}\n")
            for i in range(0, len(row.sequence), 80):
                f.write(row.sequence[i:i + 80] + "\n")


In [3]:

pep_records = load_uniprot_json(PEP_JSON)
propep_records = load_uniprot_json(PROPEP_JSON)

records_by_acc = {}
for record in pep_records + propep_records:
    acc = record.get("primaryAccession")
    if acc is None:
        print("Warning: record without primaryAccession, skipping")
        continue
    # UniProt отдаёт полный entry; если запись есть в обоих JSON, достаточно одного экземпляра.
    records_by_acc[acc] = record

records = list(records_by_acc.values())

print("pep_records:", len(pep_records))
print("propep_records:", len(propep_records))
print("unique records:", len(records))


pep_records: 8579
propep_records: 13039
unique records: 18091


In [4]:

protein_rows = []
feature_rows = []

for record in records:
    accession = record.get("primaryAccession")
    entry_name = record.get("uniProtkbId")
    sequence = get_nested(record, "sequence", "value")
    sequence_len = get_nested(record, "sequence", "length")

    if not accession or not sequence:
        continue

    organism = get_nested(record, "organism", "scientificName")
    taxon_id = get_nested(record, "organism", "taxonId")
    lineage = get_nested(record, "organism", "lineage", default=[])

    protein_name = get_nested(
        record,
        "proteinDescription",
        "recommendedName",
        "fullName",
        "value",
    )

    gene_names = []
    for gene in record.get("genes", []) or []:
        gene_name = get_nested(gene, "geneName", "value")
        if gene_name:
            gene_names.append(gene_name)

    protein_rows.append({
        "accession": accession,
        "protein_id": accession,
        "entry_name": entry_name,
        "sequence": sequence,
        "sequence_md5": sequence_md5(sequence),
        "length": int(sequence_len) if sequence_len is not None else len(sequence),
        "organism": organism,
        "taxon_id": taxon_id,
        "lineage": lineage,
        "protein_name": protein_name,
        "gene_names": gene_names,
        "is_viral": "Viruses" in lineage if isinstance(lineage, list) else False,
        "raw_entry": record,  # удобно для отладки; перед parquet удалим
    })

    for feature in record.get("features", []) or []:
        raw_type = feature.get("type")
        label = TARGET_TYPES.get(raw_type, raw_type)

        start, end, start_modifier, end_modifier = get_location(feature)

        drop_reasons = []
        feature_len = None
        feature_seq = None

        if start is None or end is None:
            drop_reasons.append("missing_location")
        else:
            feature_len = end - start + 1
            if start < 1 or end > len(sequence) or start > end:
                drop_reasons.append("location_out_of_bounds")
            else:
                feature_seq = sequence[start - 1:end]  # UniProt 1-based inclusive

        if start_modifier not in (None, "EXACT"):
            drop_reasons.append(f"non_exact_start:{start_modifier}")

        if end_modifier not in (None, "EXACT"):
            drop_reasons.append(f"non_exact_end:{end_modifier}")

        is_target_type = raw_type in TARGET_TYPES

        # Basic DeepPeptide length filter: только для PEPTIDE/PROPEP.
        if is_target_type and feature_len is not None:
            if feature_len < MIN_FEATURE_LEN:
                drop_reasons.append("too_short")
            if feature_len > MAX_FEATURE_LEN:
                drop_reasons.append("too_long")

        evidence_codes, evidence_sources, evidence_ids, pubmed_ids = extract_evidence(feature)

        feature_rows.append({
            "accession": accession,
            "entry_name": entry_name,
            "raw_feature_type": raw_type,
            "label": label,
            "start": start,
            "end": end,
            "length": feature_len,
            "feature_sequence": feature_seq,
            "description": feature.get("description"),
            "start_modifier": start_modifier,
            "end_modifier": end_modifier,
            "evidence_codes": evidence_codes,
            "evidence_sources": evidence_sources,
            "evidence_ids": evidence_ids,
            "pubmed_ids": pubmed_ids,
            "has_sorting_prorule": feature_has_sorting_prorule(feature),
            "is_target_type": is_target_type,
            "basic_drop_reason": ";".join(drop_reasons),
            "is_valid_basic": is_target_type and len(drop_reasons) == 0,
            # raw_feature не сохраняем в parquet; если нужно отлаживать PRU, можно временно раскомментировать.
            # "raw_feature": feature,
        })


proteins_master = pd.DataFrame(protein_rows).drop_duplicates("accession")
features_master = pd.DataFrame(feature_rows)

print("proteins_master:", len(proteins_master))
print("features_master:", len(features_master))
print("target feature counts, before filters:")
print(features_master.loc[features_master["is_target_type"], "label"].value_counts())
print("target feature counts, after basic filter:")
print(features_master.loc[features_master["is_valid_basic"], "label"].value_counts())


proteins_master: 18091
features_master: 254566
target feature counts, before filters:
label
PROPEP     15287
PEPTIDE    12381
Name: count, dtype: int64
target feature counts, after basic filter:
label
PEPTIDE    11379
PROPEP      9429
Name: count, dtype: int64


In [5]:

# DeepPeptide extra filters:
# 1) удалить PEPTIDE, если он покрывает весь белок;
# 2) удалить PEPTIDE, если PEPTIDE + SIGNAL/TRANSIT покрывают весь белок;
# 3) удалить PROPEP sorting signals, размеченные через PROSITE ProRules PRU00477/PRU01070.

protein_len_by_acc = proteins_master.set_index("accession")["length"].to_dict()

targeting_by_acc = defaultdict(list)
for f in features_master[features_master["raw_feature_type"].isin(TARGETING_FEATURE_TYPES)].itertuples(index=False):
    if pd.notna(f.start) and pd.notna(f.end):
        targeting_by_acc[f.accession].append((int(f.start), int(f.end), f.raw_feature_type))

deeppeptide_extra_drop_reasons = []

for f in features_master.itertuples(index=False):
    reasons = []
    protein_len = protein_len_by_acc.get(f.accession)

    if f.is_valid_basic:
        if f.label == "PEPTIDE":
            peptide_interval = (int(f.start), int(f.end))

            if protein_len is not None and peptide_interval == (1, int(protein_len)):
                reasons.append("peptide_full_length")

            intervals = [peptide_interval]
            intervals += [(s, e) for s, e, _ in targeting_by_acc.get(f.accession, [])]

            if protein_len is not None and intervals_cover_full_length(intervals, protein_len):
                reasons.append("peptide_plus_signal_or_transit_full_length")

        if f.label == "PROPEP" and bool(f.has_sorting_prorule):
            reasons.append("propeptide_sorting_signal_prorule_PRU00477_or_PRU01070")

    deeppeptide_extra_drop_reasons.append(";".join(reasons))

features_master["deeppeptide_extra_drop_reason"] = deeppeptide_extra_drop_reasons

features_master["drop_reason"] = features_master.apply(
    lambda r: ";".join(x for x in [r["basic_drop_reason"], r["deeppeptide_extra_drop_reason"]] if x),
    axis=1,
)

features_master["is_valid_for_training"] = (
    features_master["is_valid_basic"] &
    (features_master["deeppeptide_extra_drop_reason"] == "")
)

valid_features = features_master[
    features_master["is_valid_for_training"]
].copy()

valid_features = valid_features.drop_duplicates(
    subset=["accession", "label", "start", "end"]
)

valid_accessions = set(valid_features["accession"])
proteins_train = proteins_master[
    proteins_master["accession"].isin(valid_accessions)
].copy()

print("Valid training features:")
print(valid_features["label"].value_counts())
print("Valid training proteins:", len(proteins_train))

print("\nExtra DeepPeptide drop reasons:")
print(features_master.loc[
    features_master["deeppeptide_extra_drop_reason"] != "",
    "deeppeptide_extra_drop_reason"
].value_counts())

print("\nAll target drop reasons:")
print(features_master.loc[
    features_master["is_target_type"] & ~features_master["is_valid_for_training"],
    "drop_reason"
].value_counts().head(30))


Valid training features:
label
PROPEP     9140
PEPTIDE    7431
Name: count, dtype: int64
Valid training proteins: 9619

Extra DeepPeptide drop reasons:
deeppeptide_extra_drop_reason
peptide_full_length;peptide_plus_signal_or_transit_full_length    3817
propeptide_sorting_signal_prorule_PRU00477_or_PRU01070             289
peptide_plus_signal_or_transit_full_length                         130
Name: count, dtype: int64

All target drop reasons:
drop_reason
peptide_full_length;peptide_plus_signal_or_transit_full_length    3817
too_long                                                          3532
too_short                                                         2501
propeptide_sorting_signal_prorule_PRU00477_or_PRU01070             289
missing_location;non_exact_start:UNKNOWN                           227
missing_location;non_exact_end:UNKNOWN                             199
non_exact_start:OUTSIDE                                            152
peptide_plus_signal_or_transit_full_length  

In [6]:

# Сборка legacy labeled_sequences.csv в формате, максимально близком к оригинальному DeepPeptide:
# ,protein_name,sequence,organism,is_peptide,coordinates,is_propeptide,propeptide_coordinates,protein_id

legacy_rows = []

for protein in proteins_train.itertuples(index=False):
    acc = protein.accession
    feats = valid_features[valid_features["accession"] == acc]

    peptide_coords = coords_to_string(feats, "PEPTIDE")
    propeptide_coords = coords_to_string(feats, "PROPEP")

    legacy_rows.append({
        "protein_name": protein.entry_name,
        "sequence": protein.sequence,
        "organism": protein.organism,
        "is_peptide": binary_string(protein.sequence, parse_coords(peptide_coords)),
        "coordinates": peptide_coords,
        "is_propeptide": binary_string(protein.sequence, parse_coords(propeptide_coords)),
        "propeptide_coordinates": propeptide_coords,
        "protein_id": acc,
    })

labeled_sequences = pd.DataFrame(legacy_rows)

# Стабильный порядок строк.
labeled_sequences = labeled_sequences.sort_values("protein_id").reset_index(drop=True)

labeled_path = DATA_DIR / "labeled_sequences.csv"
labeled_sequences.to_csv(labeled_path, index=True)

print("Saved:", labeled_path)
print("Rows:", len(labeled_sequences))
print("With PEPTIDE:", int((labeled_sequences["coordinates"] != "").sum()))
print("With PROPEP:", int((labeled_sequences["propeptide_coordinates"] != "").sum()))
print("With both:", int(((labeled_sequences["coordinates"] != "") & (labeled_sequences["propeptide_coordinates"] != "")).sum()))
labeled_sequences.head()


Saved: /home/oskar/work/DeepPeptide/data/uniprot_2026/labeled_sequences.csv
Rows: 9619
With PEPTIDE: 4385
With PROPEP: 8007
With both: 2773


,protein_name,sequence,organism,is_peptide,coordinates,is_propeptide,propeptide_coordinates,protein_id
0,MSD4_AMAEX,MSDINATRLPVWIGYSPCVGDDCIALLTRGEGLC,Amanita exitialis,0000000000111111100000000000000000,(11-17),1111111111000000011111111111111111,"(1-10),(18-34)",A0A023IWD9
1,MSD1_AMAFL,MSDINATCLPAWLALCPCVGDDVNPTLTRGGT,Amanita fuligineoides,00000000001111111000000000000000,(11-17),11111111110000000111111111111111,"(1-10),(18-32)",A0A023IWE0
2,MSD4_AMAPH,MSDINGTRLPWLATCPCVGEDVNPTLSRGER,Amanita phalloides,0000000000111111000000000000000,(11-16),1111111111000000111111111111111,"(1-10),(17-31)",A0A023IWE1
3,BAMAT_AMAPL,MSDINATRLPIWGIGCDPCVGDDVTAVLTRGEA,Amanita pallidorosea,000000000011111111000000000000000,(11-18),111111111100000000111111111111111,"(1-10),(19-33)",A0A023IWE2
4,AAMA1_AMAFL,MSDINATRLPIWGIGCNPCVGDEVTALLTRGEA,Amanita fuligineoides,000000000011111111000000000000000,(11-18),111111111100000000111111111111111,"(1-10),(19-33)",A0A023IWE3


In [ ]:

# FASTA для GraphPart и для подсчёта эмбеддингов.
# Пишем и plural, и singular вариант, чтобы не ловить баги из-за имени файла.

fasta_plural = DATA_DIR / "protein_sequences.fasta"

write_fasta(labeled_sequences, fasta_plural)

print("Saved:", fasta_plural)

# Homo sapiens ids, если нужно отдельно оценивать human subset.
homo_ids = labeled_sequences.loc[
    labeled_sequences["organism"].fillna("").str.contains("Homo sapiens", case=False, regex=False),
    "protein_id"
].tolist()

homo_path = DATA_DIR / "protein_id_homo.txt"
with open(homo_path, "w", encoding="utf-8") as f:
    for pid in homo_ids:
        f.write(pid + "\n")

print("Saved:", homo_path, "n =", len(homo_ids))


Saved: /home/oskar/work/DeepPeptide/data/uniprot_2026/protein_sequences.fasta
Saved: /home/oskar/work/DeepPeptide/data/uniprot_2026/protein_sequence.fasta
Saved: /home/oskar/work/DeepPeptide/data/uniprot_2026/protein_id_homo.txt n = 457


In [8]:

# Перед сохранением parquet убираем raw_entry, потому что это тяжёлый вложенный dict.
proteins_master_save = proteins_master.drop(columns=["raw_entry"], errors="ignore").copy()

proteins_master_save.to_parquet(PROCESSED_DIR / "proteins_master.parquet", index=False)
features_master.to_parquet(PROCESSED_DIR / "features_master.parquet", index=False)

print("Saved:", PROCESSED_DIR / "proteins_master.parquet")
print("Saved:", PROCESSED_DIR / "features_master.parquet")


Saved: /home/oskar/work/DeepPeptide/data/uniprot_2026/processed_data/proteins_master.parquet
Saved: /home/oskar/work/DeepPeptide/data/uniprot_2026/processed_data/features_master.parquet


In [9]:

# Build report

def class_counts_from_labeled(df):
    total = Counter()
    for row in df.itertuples(index=False):
        seq = row.sequence
        labels = np.array(["OTHER"] * len(seq), dtype=object)

        # PROPEP сначала, PEPTIDE поверх — тот же приоритет, что обычно удобен для конфликтов.
        for start, end in parse_coords(row.propeptide_coordinates):
            labels[start - 1:end] = "PROPEP"
        for start, end in parse_coords(row.coordinates):
            labels[start - 1:end] = "PEPTIDE"

        total["OTHER"] += int((labels == "OTHER").sum())
        total["PEPTIDE"] += int((labels == "PEPTIDE").sum())
        total["PROPEP"] += int((labels == "PROPEP").sum())

    total["TOTAL"] = total["OTHER"] + total["PEPTIDE"] + total["PROPEP"]
    return dict(total)


residue_counts = class_counts_from_labeled(labeled_sequences)
residue_fracs = {
    k + "_frac": (v / residue_counts["TOTAL"] if residue_counts["TOTAL"] else 0.0)
    for k, v in residue_counts.items()
    if k != "TOTAL"
}

report = {
    "data_dir": str(DATA_DIR),
    "n_pep_records_raw": len(pep_records),
    "n_propep_records_raw": len(propep_records),
    "n_records_unique": len(records),
    "n_proteins_master": int(len(proteins_master_save)),
    "n_features_master": int(len(features_master)),
    "n_valid_training_features": int(len(valid_features)),
    "n_training_proteins": int(len(labeled_sequences)),
    "valid_feature_counts": {
        str(k): int(v)
        for k, v in valid_features["label"].value_counts().items()
    },
    "all_target_feature_counts": {
        str(k): int(v)
        for k, v in features_master.loc[
            features_master["is_target_type"],
            "label"
        ].value_counts().items()
    },
    "target_drop_reasons": {
        str(k): int(v)
        for k, v in features_master.loc[
            features_master["is_target_type"] & ~features_master["is_valid_for_training"],
            "drop_reason"
        ].value_counts().items()
    },
    "deeppeptide_extra_drop_reasons": {
        str(k): int(v)
        for k, v in features_master.loc[
            features_master["deeppeptide_extra_drop_reason"] != "",
            "deeppeptide_extra_drop_reason"
        ].value_counts().items()
    },
    "n_with_peptide": int((labeled_sequences["coordinates"] != "").sum()),
    "n_with_propeptide": int((labeled_sequences["propeptide_coordinates"] != "").sum()),
    "n_with_both": int(
        ((labeled_sequences["coordinates"] != "") &
         (labeled_sequences["propeptide_coordinates"] != "")).sum()
    ),
    "n_homo_sapiens": int(len(homo_ids)),
    "residue_counts": {str(k): int(v) for k, v in residue_counts.items()},
    "residue_fracs": residue_fracs,
}

report_path = PROCESSED_DIR / "build_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Saved:", report_path)
print(json.dumps(report, indent=2, ensure_ascii=False))


Saved: /home/oskar/work/DeepPeptide/data/uniprot_2026/processed_data/build_report.json
{
  "data_dir": "/home/oskar/work/DeepPeptide/data/uniprot_2026",
  "n_pep_records_raw": 8579,
  "n_propep_records_raw": 13039,
  "n_records_unique": 18091,
  "n_proteins_master": 18091,
  "n_features_master": 254566,
  "n_valid_training_features": 16571,
  "n_training_proteins": 9619,
  "valid_feature_counts": {
    "PROPEP": 9140,
    "PEPTIDE": 7431
  },
  "all_target_feature_counts": {
    "PROPEP": 15287,
    "PEPTIDE": 12381
  },
  "target_drop_reasons": {
    "peptide_full_length;peptide_plus_signal_or_transit_full_length": 3817,
    "too_long": 3532,
    "too_short": 2501,
    "propeptide_sorting_signal_prorule_PRU00477_or_PRU01070": 289,
    "missing_location;non_exact_start:UNKNOWN": 227,
    "missing_location;non_exact_end:UNKNOWN": 199,
    "non_exact_start:OUTSIDE": 152,
    "peptide_plus_signal_or_transit_full_length": 130,
    "non_exact_end:OUTSIDE": 127,
    "non_exact_start:OUTSIDE;

## GraphPart

После пересборки `labeled_sequences.csv` и `protein_sequences.fasta` надо заново построить folds. Лучше запускать из консоли из корня проекта.

Первый запуск с сохранением edge cache:

```bash
graphpart needle \
  --fasta-file data/uniprot_2026/protein_sequences.fasta \
  --threshold 0.3 \
  --partitions 5 \
  --out-file data/uniprot_2026/graphpart_assignments.raw.csv \
  --threads 12 \
  --no-moving \
  --save-checkpoint-path data/uniprot_2026/processed_data/graphpart_edges_threshold_30.csv
```

Если edge cache уже есть, можно быстро делать другие split'ы:

```bash
graphpart precomputed \
  --fasta-file data/uniprot_2026/protein_sequences.fasta \
  --edge-file data/uniprot_2026/processed_data/graphpart_edges_threshold_30.csv \
  --threshold 0.3 \
  --partitions 12 \
  --out-file data/uniprot_2026/graphpart_assignments_12.raw.csv \
  --no-moving
```


In [10]:

# Нормализация GraphPart output к формату DeepPeptide: AC,cluster.
# Запусти после того, как graphpart_assignments.raw.csv появится.

raw_gp_path = DATA_DIR / "graphpart_assignments.raw.csv"
final_gp_path = DATA_DIR / "graphpart_assignments.csv"

if raw_gp_path.exists():
    gp = pd.read_csv(raw_gp_path)
    print("Raw GraphPart columns:", gp.columns.tolist())

    if "AC" not in gp.columns:
        # Иногда GraphPart может назвать колонку иначе.
        candidates = [c for c in gp.columns if c.lower() in {"ac", "id", "name", "sequence_id", "protein_id"}]
        if not candidates:
            raise ValueError(f"Не могу найти колонку ID в GraphPart output: {gp.columns.tolist()}")
        gp = gp.rename(columns={candidates[0]: "AC"})

    if "cluster" not in gp.columns:
        raise ValueError(f"Не могу найти колонку cluster в GraphPart output: {gp.columns.tolist()}")

    gp_out = gp[["AC", "cluster"]].copy()
    gp_out["cluster"] = gp_out["cluster"].astype(int)
    gp_out.to_csv(final_gp_path, index=False)

    print("Saved:", final_gp_path)
    print(gp_out["cluster"].value_counts().sort_index())
    print("Assigned proteins:", len(gp_out))
else:
    print("No raw GraphPart file yet:", raw_gp_path)


No raw GraphPart file yet: /home/oskar/work/DeepPeptide/data/uniprot_2026/graphpart_assignments.raw.csv


In [11]:

# Sanity checks для labeled + graphpart.

gp_path = DATA_DIR / "graphpart_assignments.csv"

if gp_path.exists():
    labeled = pd.read_csv(DATA_DIR / "labeled_sequences.csv")
    gp = pd.read_csv(gp_path)

    common = set(labeled["protein_id"]) & set(gp["AC"])
    print("labeled:", len(labeled))
    print("graphpart:", len(gp))
    print("common:", len(common))
    print("missing in graphpart:", labeled["protein_id"].nunique() - len(common))
    print("fold sizes:")
    print(gp[gp["AC"].isin(set(labeled["protein_id"]))]["cluster"].value_counts().sort_index())

    # Class balance by fold.
    rows = []
    labeled_by_id = labeled.set_index("protein_id")

    for r in gp.itertuples(index=False):
        if r.AC not in labeled_by_id.index:
            continue

        row = labeled_by_id.loc[r.AC]
        seq = row["sequence"]
        labels = np.array(["OTHER"] * len(seq), dtype=object)

        for start, end in parse_coords(row["propeptide_coordinates"]):
            labels[start - 1:end] = "PROPEP"
        for start, end in parse_coords(row["coordinates"]):
            labels[start - 1:end] = "PEPTIDE"

        rows.append({
            "protein_id": r.AC,
            "cluster": int(r.cluster),
            "n_residues": len(seq),
            "OTHER": int((labels == "OTHER").sum()),
            "PEPTIDE": int((labels == "PEPTIDE").sum()),
            "PROPEP": int((labels == "PROPEP").sum()),
        })

    cb = pd.DataFrame(rows)
    summary = cb.groupby("cluster").agg(
        n_proteins=("protein_id", "count"),
        total_residues=("n_residues", "sum"),
        OTHER=("OTHER", "sum"),
        PEPTIDE=("PEPTIDE", "sum"),
        PROPEP=("PROPEP", "sum"),
    ).reset_index()

    for label in ["OTHER", "PEPTIDE", "PROPEP"]:
        summary[f"{label}_frac"] = summary[label] / summary["total_residues"]

    display(summary)
    summary.to_csv(PROCESSED_DIR / "graphpart_assignments.class_balance.csv", index=False)
else:
    print("No graphpart_assignments.csv yet:", gp_path)


labeled: 9619
graphpart: 12833
common: 9382
missing in graphpart: 237
fold sizes:
cluster
0.0    1979
1.0    2469
2.0    2140
3.0    2421
4.0     373
Name: count, dtype: int64


,cluster,n_proteins,total_residues,OTHER,PEPTIDE,PROPEP,OTHER_frac,PEPTIDE_frac,PROPEP_frac
0,0,1979,155085,72916,40004,42165,0.470168,0.257949,0.271883
1,1,2469,752056,675237,34240,42579,0.897855,0.045529,0.056617
2,2,2140,568521,505867,16886,45768,0.889795,0.029702,0.080504
3,3,2421,672452,585181,37817,49454,0.870220,0.056237,0.073543
4,4,373,25461,11777,7900,5784,0.462551,0.310278,0.227171
